# **Notebook 1: Data Understanding & Exploratory Data Analysis**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED for this notebook:**
- [ ] `corporate_policies/` folder with `.md` SOP files uploaded to Colab:
  - `refund_policy.md`, `shipping_delays.md`, `password_reset.md`, `technical_troubleshooting.md`, `escalation_matrix.md`
- [ ] Internet access (to download Bitext dataset from HuggingFace)

**Files this notebook will CREATE:**
- [ ] `sampled_data.csv` — Cleaned, deduplicated, 4000-row sample of the Bitext dataset
  _(Required by: Notebook 2 — Data Preparation)_

> **Note:** Tasks 1.1 (Define Project Scope) and 1.4 (Design Methodology) are documented in your **Project Proposal PDF**, not this notebook. This notebook covers Tasks **1.2** and **1.3**.

---

### Setup
Installs dependencies, defines helpers and (for Notebooks 3-7) loads the shared `config.json` written by Notebook 2. Run this cell first.

In [ ]:
# ---- Environment setup (Google Colab / local Jupyter) ----
import os, sys, glob, zipfile, json, random, warnings
warnings.filterwarnings("ignore")

USE_DRIVE = False   # set True on Colab to keep artifacts (chroma_db, adapters, csv) across runtime resets
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/GenAI_Capstone"
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

%pip install -q datasets pandas numpy scikit-learn matplotlib seaborn

def find_sop_dir():
    """Locate the folder holding the Markdown SOP files (corporate_policies/ or Dataset/sop_documents/).
    If it is not present, try to unzip a Dataset*.zip, and finally ask for an upload on Colab."""
    def _search():
        for c in ["corporate_policies", os.path.join("Dataset", "sop_documents"), "sop_documents"]:
            if os.path.isdir(c) and glob.glob(os.path.join(c, "*.md")):
                return c
        for pat in ["**/sop_documents", "**/corporate_policies"]:
            for h in glob.glob(pat, recursive=True):
                if glob.glob(os.path.join(h, "*.md")):
                    return h
        return None
    found = _search()
    if found: return found
    for z in glob.glob("*.zip") + glob.glob("**/Dataset*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf: zf.extractall(".")
        found = _search()
        if found: return found
    try:
        from google.colab import files
        print("SOP folder not found - please upload Dataset.zip (contains sop_documents/*.md)")
        for name in files.upload():
            if name.endswith(".zip"):
                with zipfile.ZipFile(name) as zf: zf.extractall(".")
        found = _search()
        if found: return found
    except ImportError:
        pass
    raise FileNotFoundError("Could not find the SOP .md files. Place them in ./corporate_policies/ or ./Dataset/sop_documents/")

## **Stage 1: Data Understanding & EDA**
### **Task 1.2: Understand and Assess Data**

#### **1.2.1 Inspect Dataset Structure [2 marks]**
**The Task:** Identify the presence of the corporate policy files within the local environment and successfully download the `bitext` customer support dataset from Hugging Face. Print the dataset information.

**Hints & Tips:**
* Use `glob` to find `.md` files in your workspace, and `load_dataset` for the bitext data.
* The dataset ID is `bitext/Bitext-customer-support-llm-chatbot-training-dataset` — it contains ~26,872 rows.
* Key columns: `instruction` (user query), `intent` (target class), `category` (broader group).

**Why we are doing it:** To ensure the local environment is properly configured with our knowledge base before reading data.
**How we are doing it:** Check the `corporate_policies` directory, load the dataset, and use `.info()`.

**Learner Inference:** If the files load correctly, your environment is stable. Seeing the 'intent' column confirms the target variable you will train your model to extract in Stage 4.

In [ ]:
import glob, os, re, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

# --- (a) corporate policy (SOP) files -----------------------------------------------------------
SOP_DIR = find_sop_dir()
sop_files = sorted(glob.glob(os.path.join(SOP_DIR, "*.md")))
print(f"SOP folder: {SOP_DIR}  ->  {len(sop_files)} Markdown files")
required = ["refund_policy.md", "shipping_delays.md", "password_reset.md",
            "technical_troubleshooting.md", "escalation_matrix.md"]
missing = [r for r in required if r not in {os.path.basename(f) for f in sop_files}]
assert not missing, f"Missing mandatory SOP files: {missing}"
sop_summary = []
for f in sop_files:
    txt = open(f, encoding="utf-8").read()
    sop_summary.append({"file": os.path.basename(f), "title": txt.splitlines()[0].lstrip("# ").strip(),
                        "words": len(txt.split()), "sections(##)": len(re.findall(r"(?m)^## ", txt))})
display(pd.DataFrame(sop_summary))

# --- (b) Bitext customer-support dataset --------------------------------------------------------
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
print(dataset)
df_raw = dataset["train"].to_pandas()
print("\n--- df_raw.info() ---")
df_raw.info()
display(df_raw[["instruction", "intent", "category"]].head())

# --- (c) completeness check ---------------------------------------------------------------------
key_cols = ["instruction", "intent", "category"]
print("\nNull values per key column:\n", df_raw[key_cols].isnull().sum())
print("Empty-string values per key column:\n", (df_raw[key_cols].astype(str).apply(lambda s: s.str.strip() == "")).sum())
print("\nLearner inference: SOP files loaded =", len(sop_files), "| target variable 'intent' present =", "intent" in df_raw.columns)

#### **1.2.2 Quantify Dataset Characteristics [3 marks]**
**The Task:** Calculate the original size of the dataset, identify the number of unique intents, and randomly sample the dataset down to 4,000 rows.

**Hints & Tips:**
* Use `len()`, `.nunique()`, and `.sample(4000, random_state=42)`.
* `random_state=42` ensures reproducibility — everyone gets the same sample.
* With 4000 rows and 27 intents, you'll get ~148 examples per intent on average.

**Parameter Tuning:**
* `sample(n)` — try 1000–5000 per the workflow spec:
  - `1000`: Fast (~1 min), lower accuracy
  - `4000`: Moderate (~3 min), better accuracy (recommended)
  - `5000`: Slower, best accuracy, may hit T4 memory limits

**Learner Inference:** Capping at 4000 rows keeps your Stage 4 training loop to minutes while providing enough examples per intent.

In [ ]:
original_size = len(df_raw)
n_intents = df_raw["intent"].nunique()
n_categories = df_raw["category"].nunique()
print(f"Original dataset size : {original_size:,} rows")
print(f"Unique intents        : {n_intents}")
print(f"Unique categories     : {n_categories}")

# Prompt-length characteristics of the FULL dataset (token-level analysis follows in Notebook 2)
df_raw["n_words"] = df_raw["instruction"].str.split().str.len()
df_raw["n_chars"] = df_raw["instruction"].str.len()
print("\nPrompt length statistics (full dataset):")
display(df_raw[["n_words", "n_chars"]].describe().T.round(1))

# Down-sample to 4,000 rows (inside the mandated 1,000-5,000 range). random_state=42 -> reproducible.
SAMPLE_SIZE = 4000
df_sample = df_raw.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"\nSampled rows: {len(df_sample):,}  |  intents present: {df_sample['intent'].nunique()}")
per_intent = df_sample["intent"].value_counts()
print(f"Examples per intent -> mean {per_intent.mean():.0f}, min {per_intent.min()}, max {per_intent.max()}")
assert 1000 <= len(df_sample) <= 5000

#### **1.2.3 Validate Sample Quality [2 marks]**
**The Task:** Identify and remove records containing null values or duplicated customer instructions using normalised deduplication.

**Hints & Tips:**
* Use `.isnull().sum()` to check nulls.
* Simple `drop_duplicates` misses near-dupes like `"Where is my order?"` vs `"where is my order?"`.
* Normalise to lowercase + strip whitespace BEFORE deduplicating to catch these.
* The Bitext dataset typically has ~15 near-duplicates in a 4000-row sample.

**Learner Inference:** Cleaning now prevents "garbage-in, garbage-out". Duplicate prompts can cause leakage between train and test sets later.

In [ ]:
key_cols = ["instruction", "intent", "category"]
df = df_sample.copy()

# 1) nulls / empty strings ---------------------------------------------------------------------------
print("Nulls per key column:\n", df[key_cols].isnull().sum())
empty_mask = df[key_cols].astype(str).apply(lambda s: s.str.strip() == "").any(axis=1)
print("Rows with an empty-string key field:", int(empty_mask.sum()))

# 2) formatting inconsistencies ----------------------------------------------------------------------
ws_issue = (df["instruction"] != df["instruction"].str.strip()).sum()
case_issue_intent = (df["intent"] != df["intent"].str.lower()).sum()
case_issue_cat = (df["category"] != df["category"].str.upper()).sum()
print(f"Whitespace-padded prompts: {ws_issue} | intents not lower-case: {case_issue_intent} | categories not UPPER: {case_issue_cat}")

# 3) exact vs NORMALISED duplicates (lower-case + strip BEFORE comparison) ---------------------------
exact_dups = df.duplicated(subset=["instruction"]).sum()
df["_norm"] = df["instruction"].astype(str).str.lower().str.strip()
norm_dups = df.duplicated(subset=["_norm"]).sum()
print(f"Exact duplicates: {exact_dups} | normalised duplicates: {norm_dups} (extra near-duplicates caught: {norm_dups - exact_dups})")

before = len(df)
df = df.dropna(subset=key_cols)
df = df[~df[key_cols].astype(str).apply(lambda s: s.str.strip() == "").any(axis=1)]
df = df.drop_duplicates(subset="_norm").drop(columns="_norm")
print(f"Rows removed: {before - len(df)}  ({before} -> {len(df)})")

# 4) prompt-solution alignment: every intent must map to exactly ONE category -------------------------
align = df.groupby("intent")["category"].nunique()
print("\nIntents mapped to more than one category:", int((align > 1).sum()))
assert (align == 1).all(), "Intent -> category alignment broken"
print("Prompt-solution alignment OK: every intent belongs to exactly one category.")

df_clean = df.reset_index(drop=True)
print(f"\nClean sample: {len(df_clean):,} rows | intents: {df_clean['intent'].nunique()} | categories: {df_clean['category'].nunique()}")
assert 1000 <= len(df_clean) <= 5000
print("Learner inference: cleaning now prevents duplicate prompts from leaking across train/test later.")

### **Task 1.3: Perform Exploratory Data Analysis**

#### **1.3.1 Perform Univariate Analysis [2 marks]**
**The Task:** Plot the top 10 most frequent customer intents using a horizontal bar chart.

**Hints & Tips:**
* Use a Seaborn `countplot` with `order=df['intent'].value_counts().index[:10]`.
* Check if any single intent dominates >80% of data — signals severe class imbalance.
* `palette="viridis"` works well for accessibility.

**Learner Inference:** This visual reveals class imbalance. If one intent dominates, your fine-tuned router becomes biased toward guessing it.

In [ ]:
top10 = df_clean["intent"].value_counts().index[:10]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=df_clean, y="intent", hue="intent", order=top10, palette="viridis", legend=False, ax=axes[0])
axes[0].set_title("Top 10 most frequent customer intents"); axes[0].set_xlabel("Number of prompts")

df_clean["n_words"] = df_clean["instruction"].str.split().str.len()
sns.histplot(df_clean["n_words"], bins=30, color="#3b6fb6", ax=axes[1])
axes[1].set_title("Prompt length (words)"); axes[1].set_xlabel("Words per prompt")
plt.tight_layout(); plt.show()

vc = df_clean["intent"].value_counts()
top_share = vc.iloc[0] / len(df_clean) * 100
imb_ratio = vc.iloc[0] / vc.iloc[-1]
print(f"Most frequent intent: '{vc.index[0]}' = {top_share:.1f}% of data | least frequent: '{vc.index[-1]}' ({vc.iloc[-1]})")
print(f"Imbalance ratio (max/min): {imb_ratio:.2f}")
if top_share > 80:
    print("WARNING: severe class imbalance (>80%) - the fine-tuned router would be biased.")
else:
    print("Interpretation: no single intent dominates (<80%); class balance is acceptable for training the router.")
print(f"Prompt length: mean {df_clean['n_words'].mean():.1f} words, 95th percentile {df_clean['n_words'].quantile(.95):.0f} words -> short prompts.")

#### **1.3.2 Perform Bivariate Analysis [2 marks]**
**The Task:** Read the text from the uploaded Markdown SOP files and fit a `TfidfVectorizer` (with `stop_words='english'`) to transform them into vectors.

**Hints & Tips:**
* Read files into a list of strings first using `open(f, 'r').read()`.
* `TfidfVectorizer(stop_words="english")` removes common words, keeping meaningful terms.
* The resulting matrix shape should be `(num_docs, num_unique_terms)`.

**Learner Inference:** This transforms raw text into math, letting you prove whether your policies are distinct enough for an AI to tell apart.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

sop_names, sop_texts = [], []
for f in sorted(glob.glob(os.path.join(SOP_DIR, "*.md"))):
    with open(f, "r", encoding="utf-8") as fh:
        sop_texts.append(fh.read())
    sop_names.append(os.path.basename(f).replace(".md", ""))

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(sop_texts)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}  (num_docs, num_unique_terms)")

terms = np.array(tfidf.get_feature_names_out())
rows = []
for i, name in enumerate(sop_names):
    top = terms[np.argsort(-tfidf_matrix[i].toarray()[0])[:6]]
    rows.append({"SOP": name, "top distinguishing terms": ", ".join(top)})
display(pd.DataFrame(rows))
print("Learner inference: each SOP has its own vocabulary -> a retriever has signal to tell them apart.")

#### **1.3.3 Generate Visualisations [2 marks]**
**The Task:** Calculate cosine similarity between the TF-IDF vectors of the SOP documents and plot an annotated heatmap.

**Hints & Tips:**
* Use `cosine_similarity` from sklearn and `sns.heatmap` with `annot=True`.
* Diagonal should always be 1.0 (document vs itself).
* Off-diagonal >0.5 = high overlap (RAG may struggle); <0.3 = low overlap (good retrieval).

**Learner Inference:** High overlap (dark off-diagonal squares) means the RAG system struggles to fetch the right doc. Low overlap proves retrieval will be accurate.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(tfidf_matrix)
sim_df = pd.DataFrame(sim, index=sop_names, columns=sop_names)
plt.figure(figsize=(11, 9))
sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, square=True, cbar_kws={"label": "cosine similarity"})
plt.title("TF-IDF cosine similarity between SOP documents"); plt.tight_layout(); plt.show()

assert np.allclose(np.diag(sim), 1.0)
iu = np.triu_indices_from(sim, k=1)
pairs = sorted(zip(sim[iu], np.array(sop_names)[iu[0]], np.array(sop_names)[iu[1]]), reverse=True)
print("Most similar SOP pairs:")
for s, a, b in pairs[:5]:
    print(f"  {s:.2f}  {a}  <->  {b}")
off = sim[iu]
print(f"\nOff-diagonal similarity: mean {off.mean():.2f}, max {off.max():.2f}")
n_high = int((off > 0.5).sum()); n_mid = int(((off >= 0.3) & (off <= 0.5)).sum())
print(f"Pairs >0.5 (high overlap): {n_high} | pairs 0.3-0.5 (moderate): {n_mid}")
if n_high:
    print("Interpretation: some SOPs overlap strongly -> naive RAG may fetch the wrong policy for ambiguous queries (e.g. shipping_delays vs order_tracking).")
else:
    print("Interpretation: SOPs are lexically distinct, but SHORT NOISY customer queries share little vocabulary with them - retrieval can still fail on ambiguous wording.")

---
## Save Artifacts for Notebook 2

**IMPORTANT:** This cell saves the cleaned DataFrame. Notebook 2 depends on this file.

In [ ]:
out_cols = ["instruction", "intent", "category"]
df_clean[out_cols].to_csv("sampled_data.csv", index=False)
chk = pd.read_csv("sampled_data.csv")
print("Saved sampled_data.csv:", chk.shape, "| columns:", list(chk.columns))
print("corporate policies still accessible:", len(glob.glob(os.path.join(SOP_DIR, '*.md'))), "files in", SOP_DIR)

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify all items before proceeding to Notebook 2.**

- [ ] Bitext dataset loaded and `.info()` printed
- [ ] Dataset sampled to 4000 rows (or chosen size 1000–5000)
- [ ] Nulls inspected, near-duplicates removed via normalised dedup
- [ ] Top 10 intent bar chart rendered
- [ ] TF-IDF vectorisation completed on SOP documents
- [ ] Cosine similarity heatmap rendered
- [ ] **`sampled_data.csv` saved to disk** ← _CRITICAL for Notebook 2_
- [ ] `corporate_policies/` folder still accessible ← _Needed in Notebooks 2 and 4_

**If any item is unchecked, fix it before moving on.**